# Protein Filtering

Loads each patient's H5 file from preprocessing, removes cells with abnormally high total protein counts, exports protein-filtered H5 files, and splits the multiplexed BAM file per patient.

Method details and schematics: [README.md](README.md)

---

## Setup

### Load modules

Standard library, scientific stack, plotting and interactive tables, then Mission Bio Mosaic.
`pysam` is optional: it is only needed for BAM demultiplexing of multiplexed experiments.
Empty entries are removed from `sys.path` before importing Mosaic, as they cause a `FileNotFoundError` in setuptools.

In [ ]:
import os
import random
import shutil
import subprocess
import sys

import itables.options as opt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.offline as pyo
import seaborn as sns
import yaml
from itables import init_notebook_mode

try:
    import pysam
except ImportError:
    pysam = None

sys.path = [p for p in sys.path if p]
import missionbio.mosaic as ms

### Notebook display settings

Plotly renders offline, every DataFrame is shown as an interactive table with no row or column limits, and matplotlib plots inline.

In [ ]:
pyo.init_notebook_mode()

init_notebook_mode(all_interactive=True)
opt.lengthMenu = [5, 10, 20, 50, 100, 200, 500]
opt.maxBytes = 0
opt.maxColumns = 0

%matplotlib inline

### Reproducibility

Fixed seeds for Python and NumPy. `PYTHONHASHSEED` does not change hashing in the running kernel, but processes started later inherit it.

In [ ]:
%env PYTHONHASHSEED=42
random.seed(42)
np.random.seed(42)

## Experiment configuration

Set the experiment name below; everything else is read from `experiment_config.yml`.

- `EXPERIMENT`: experiment folder name under `metadata/` and `processed_data/h5/`.
- `SKIP_CHECK`: `True` skips the interactive confirmation.
- `SINTO_BIN`: path to the `sinto` executable used for BAM demultiplexing (multiplexed experiments only).

In [ ]:
EXPERIMENT = "pMDS001"

SKIP_CHECK = True

SINTO_BIN = "/project/meadlab/kgurashi/miniconda3/envs/sinto/bin/sinto"

### Load `experiment_config.yml`

The project root (`wd`) is two levels up from `notebooks/02_protein_filtering/`.

In [ ]:
wd = os.path.dirname(os.path.dirname(os.getcwd()))

config_path = os.path.join(wd, "metadata", EXPERIMENT, "experiment_config.yml")
with open(config_path) as f:
    config = yaml.safe_load(f)

IS_MULTIPLEXED = config.get("multiplexed", False)
N_DONORS       = config.get("n_samples_multiplexed", 1)

print(f"Experiment:  {EXPERIMENT}")
print(f"Multiplexed: {IS_MULTIPLEXED} ({N_DONORS} donor(s))")

## Load per-patient H5 files

Reads every patient folder written by the preprocessing export: `processed_data/h5/{EXPERIMENT}/demultiplexed/{patient}/` for multiplexed experiments, `processed_data/h5/{EXPERIMENT}/{patient}/` otherwise.
In each folder the first `.h5` file whose name contains `protein_unfiltered` is loaded (any `.h5` file if none matches); folders without H5 files are skipped. All variants and cells are kept.

In [ ]:
h5_root = os.path.join(wd, "processed_data", "h5", EXPERIMENT)
h5_base = os.path.join(h5_root, "demultiplexed") if IS_MULTIPLEXED else h5_root

samples = {}

for patient_id in sorted(os.listdir(h5_base)):
    patient_dir = os.path.join(h5_base, patient_id)
    if not os.path.isdir(patient_dir):
        continue

    h5_files = [f for f in os.listdir(patient_dir)
                if f.endswith(".h5") and "protein_unfiltered" in f.lower()]

    if not h5_files:
        h5_files = [f for f in os.listdir(patient_dir) if f.endswith(".h5")]

    if not h5_files:
        print(f"  ⚠ {patient_id}: no H5 files found, skipping")
        continue

    h5_path = os.path.join(patient_dir, h5_files[0])
    sample = ms.load(h5_path, filter_variants=False, filter_cells=False)
    samples[patient_id] = sample
    print(f"  ✓ {patient_id}: {sample.dna.shape[0]} cells, "
          f"{sample.protein.shape[1] if sample.protein is not None else 0} protein features "
          f"← {h5_files[0]}")

print(f"\nLoaded {len(samples)} patient(s)")

### Data overview

In [ ]:
for patient_id, sample in samples.items():
    print(f"\n── {patient_id} ──")
    for name, assay in [("DNA", sample.dna), ("Protein", sample.protein)]:
        if assay is not None:
            print(f"  {name}: {assay}")
            print(f"    row_attrs: {list(assay.row_attrs.keys())}")
            print(f"    col_attrs: {list(assay.col_attrs.keys())}")
            print(f"    layers:    {list(assay.layers.keys())}")

## Protein filtering

Removes cells with abnormally high total protein counts (likely debris or aggregates), per patient:

1. Total protein reads per cell = sum of the protein `read_counts` layer across all antibodies.
2. Threshold = Q75 + 3 × IQR of those totals (IQR = Q75 − Q25). There is no lower cut-off.
3. Cells with total reads above the threshold are removed.

Violin plots show total reads per cell before (left) and after (right) filtering. Patients without a protein assay are kept unchanged.

### Filter helpers

- `_total_protein_reads`: total protein reads per cell.
- `_reads_frame`: the same totals as a table for the violin plots.

In [ ]:
def _total_protein_reads(sample):
    return sample.protein.get_attribute("read_counts", constraint="row+col").sum(axis=1)


def _reads_frame(total_reads, patient_id):
    return pd.DataFrame({
        "Total_reads_Protein_per_Cell": total_reads,
        "Sample": patient_id,
    })

### Filter each patient

In [ ]:
filtered_samples = {}

for patient_id, sample in samples.items():
    print(f"\n{'='*50}")
    print(f"{patient_id}")
    print(f"{'='*50}")

    if sample.protein is None:
        print("  ⚠ No protein assay, skipping")
        filtered_samples[patient_id] = sample
        continue

    total_reads = _total_protein_reads(sample)
    total_reads_df = _reads_frame(total_reads, patient_id)

    stats_pre = total_reads_df["Total_reads_Protein_per_Cell"].describe(
        percentiles=[.10, .25, .5, .75, .90]
    )
    print(f"  Pre-filter:  {len(total_reads_df)} cells")
    print(f"    median={stats_pre['50%']:.0f}, Q25={stats_pre['25%']:.0f}, Q75={stats_pre['75%']:.0f}")

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.violinplot(data=total_reads_df, x="Sample", y="Total_reads_Protein_per_Cell", ax=axes[0])
    axes[0].set_title(f"{patient_id} — pre-filter")

    q25 = stats_pre['25%']
    q75 = stats_pre['75%']
    iqr = q75 - q25
    upper_threshold = q75 + 3 * iqr

    print(f"    IQR={iqr:.0f}, upper threshold={upper_threshold:.0f}")

    keep_mask = total_reads <= upper_threshold
    keep_barcodes = total_reads_df.index[keep_mask].tolist()
    filtered_sample = sample[keep_barcodes]

    n_removed = len(total_reads_df) - len(keep_barcodes)
    print(f"  Post-filter: {len(keep_barcodes)} cells (removed {n_removed})")

    total_reads_post_df = _reads_frame(_total_protein_reads(filtered_sample), patient_id)
    sns.violinplot(data=total_reads_post_df, x="Sample", y="Total_reads_Protein_per_Cell", ax=axes[1])
    axes[1].set_title(f"{patient_id} — post-filter")

    plt.tight_layout()
    plt.show()

    filtered_samples[patient_id] = filtered_sample

⚠️ **CHECKPOINT** — inspect violin plots before continuing.

In [ ]:
if not SKIP_CHECK:
    response = input("Continue? (y/n): ").strip().lower()
    if response not in ('y', 'yes'):
        raise SystemExit("Stopped by user.")
    print("✓ Continuing")
else:
    print("⚠️  SKIP_CHECK=True — continuing")

## Export protein-filtered H5

Saves each patient's filtered sample as `{patient}_protein_filtered.dna+protein.h5` in the same folder as its unfiltered file, retrying once after removing any partial file.

In [ ]:
for patient_id, sample in filtered_samples.items():
    if IS_MULTIPLEXED:
        out_dir = os.path.join(h5_root, "demultiplexed", patient_id)
    else:
        out_dir = os.path.join(h5_root, patient_id)

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{patient_id}_protein_filtered.dna+protein.h5")

    try:
        ms.save(sample, out_path)
        dna_n = sample.dna.shape[0]
        prot_n = sample.protein.shape[1] if sample.protein is not None else 0
        print(f"  ✓ {patient_id}: {dna_n} cells, {prot_n} protein features → {out_path}")
    except Exception as e:
        print(f"  ⚠ {patient_id}: save failed ({e}), retrying...")
        try:
            if os.path.exists(out_path):
                os.remove(out_path)
            ms.save(sample, out_path)
            print(f"  ✓ {patient_id}: saved on retry")
        except Exception as e2:
            print(f"  ✗ {patient_id}: failed — {e2}")

print("\nDone.")

## BAM demultiplexing

For multiplexed experiments, splits the experiment-level BAM file into one BAM per patient with `sinto filterbarcodes`. Requires `pysam` and the `sinto` executable at `SINTO_BIN`. Skipped for single-donor experiments.

1. Uses the first `.bam` file in `raw_data/bam/{EXPERIMENT}/multiplexed/`, or in `raw_data/bam/{EXPERIMENT}/` if that subfolder does not exist. **If no BAM file is found, this step is skipped** with a message, and the notebook continues. The BAM is indexed if no `.bai` exists.
2. Writes a barcode → patient table, `results/demultiplexing/{EXPERIMENT}/bam_demultiplexing_barcodes.tsv`, from the **protein-filtered** cells. A `-1` suffix is added to barcodes that lack it, as `sinto` expects.
3. Runs `sinto filterbarcodes` on the `RG` tag with (CPU count − 2) processes, writing to `raw_data/bam/{EXPERIMENT}/demultiplexed/`.
4. Moves each patient BAM into its own subfolder, `demultiplexed/{patient}/{patient}.bam`, and indexes it.

In [ ]:
BAM_PATH = None

if not IS_MULTIPLEXED:
    print("Single donor — BAM demultiplexing skipped")
else:
    bam_dir = os.path.join(wd, "raw_data", "bam", EXPERIMENT)
    mux_dir = os.path.join(bam_dir, "multiplexed")
    bam_search = mux_dir if os.path.isdir(mux_dir) else bam_dir

    bam_files = [f for f in os.listdir(bam_search) if f.endswith(".bam")] if os.path.isdir(bam_search) else []
    if bam_files:
        BAM_PATH = os.path.join(bam_search, bam_files[0])
        print(f"Multiplexed BAM: {BAM_PATH}")
    else:
        print(f"No BAM file found in {bam_search} — BAM demultiplexing skipped")

### Split the BAM per patient

Runs only when a multiplexed BAM file was found above (steps 1–4).

In [ ]:
if BAM_PATH is not None:
    if pysam is None:
        raise ImportError("pysam is required for BAM demultiplexing")

    bai_path = BAM_PATH + ".bai"
    if not os.path.exists(bai_path):
        print("  Indexing BAM...")
        pysam.index(BAM_PATH)
        print("  ✓ Indexed")
    else:
        print("  ✓ Index exists")

    barcode_tsv_path = os.path.join(
        wd, "results", "demultiplexing", EXPERIMENT, "bam_demultiplexing_barcodes.tsv"
    )
    os.makedirs(os.path.dirname(barcode_tsv_path), exist_ok=True)

    rows = []
    for patient_id, sample in filtered_samples.items():
        barcodes = np.asarray(sample.dna.row_attrs["barcode"]).astype(str)
        for bc in barcodes:
            bc_sinto = bc if bc.endswith("-1") else bc + "-1"
            rows.append(f"{bc_sinto}\t{patient_id}")

    with open(barcode_tsv_path, "w") as f:
        f.write("\n".join(rows))

    print(f"  Barcode TSV: {len(rows)} barcodes across {len(filtered_samples)} patients")
    print(f"  Written: {barcode_tsv_path}")

    demux_bam_dir = os.path.join(wd, "raw_data", "bam", EXPERIMENT, "demultiplexed")
    os.makedirs(demux_bam_dir, exist_ok=True)
    n_cores = max(1, os.cpu_count() - 2)

    assert os.path.exists(SINTO_BIN), f"sinto binary not found at {SINTO_BIN}"

    sinto_cmd = [
        SINTO_BIN, "filterbarcodes",
        "-b", BAM_PATH,
        "-c", barcode_tsv_path,
        "-p", str(n_cores),
        "--outdir", demux_bam_dir,
        "--barcodetag", "RG",
    ]
    print(f"  Running: {' '.join(sinto_cmd)}")
    result = subprocess.run(sinto_cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"  ⚠ sinto failed:\n{result.stderr}")
    else:
        print("  ✓ sinto completed")

    for patient_id in filtered_samples.keys():
        src_bam = os.path.join(demux_bam_dir, f"{patient_id}.bam")
        if not os.path.exists(src_bam):
            print(f"  ⚠ {patient_id}: BAM not found at {src_bam}")
            continue

        dst_dir = os.path.join(demux_bam_dir, patient_id)
        os.makedirs(dst_dir, exist_ok=True)
        dst_bam = os.path.join(dst_dir, f"{patient_id}.bam")

        shutil.move(src_bam, dst_bam)
        print(f"  ✓ {patient_id}: moved to {dst_bam}")

        pysam.index(dst_bam)
        print(f"    ✓ Indexed")

    print("\nBAM demultiplexing complete.")